## Importy

In [90]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

In [91]:
df = pd.read_csv('dice_ext_proj.csv')
df

,prawdziwe,oczka,2,3,4,5,6,7,8,9,...,776,777,778,779,780,781,782,783,784,785
0,0,8,220,220,220,220,220,220,220,220,...,220,220,220,220,220,220,220,220,220,220
1,0,8,235,235,235,235,235,235,235,235,...,235,235,235,235,235,235,235,235,235,235
2,0,0,5,5,5,5,5,5,5,5,...,5,5,5,5,5,5,5,5,5,5
3,0,3,49,49,49,49,49,49,49,49,...,49,49,49,49,49,49,49,49,49,49
4,0,6,56,56,56,56,56,56,56,56,...,56,56,56,56,56,56,56,56,56,56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,1,6,227,227,227,227,227,227,227,227,...,227,227,227,227,227,227,227,227,227,227
69996,1,6,243,243,243,243,243,243,243,243,...,243,243,243,243,243,243,243,243,243,243
69997,1,6,236,236,236,236,236,236,236,236,...,236,236,236,236,236,236,236,236,236,236
69998,1,6,8,8,8,8,8,8,8,8,...,8,8,8,8,8,8,8,8,8,8


In [92]:
# Przygotowanie danych
X = df.iloc[:, 2:].values.astype('float32')  # Piksle 28x28, 784 wartości, od kolumny trzeciej
y = df.iloc[:, 1].values  # Liczba oczek (od 0 do 9), kolumna 2, etykieta

In [93]:
# Normalizacja pikseli (skalowanie wartości do zakresu 0-1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [94]:
# Podział danych na treningowe i testowe (80% treningowe, 20% testowe)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [95]:
# Konwersja danych do tensora
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [96]:
# Tworzenie DataLoadera
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [97]:
# Ładowani edanych treningowych (batch_size - rozmiar, shuffle - tasowanie)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [98]:
# Definicja modelu
class ModelOczek(nn.Module):
    def __init__(self):
        super(ModelOczek, self).__init__()
        self.fc1 = nn.Linear(784, 128)  # Wejście (784 piksele) do warstwy ukrytej (128 neuronów)
        self.fc2 = nn.Linear(128, 64)   # Druga warstwa ukryta (64 neurony)
        self.fc3 = nn.Linear(64, 10)    # Wyjście (10 klas, od 0 do 9 oczek)

    def forward(self, x):
        x = torch.relu(self.fc1(x))  # Aktywacja ReLU
        x = torch.relu(self.fc2(x))  # Aktywacja ReLU
        x = self.fc3(x)  # Wyjście bez aktywacji
        return x

In [99]:
# Inicjalizacja modelu
model = ModelOczek()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ModelOczek(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)

In [100]:
# Optymalizator i funkcja strat
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()  # Używamy funkcji strat dla wieloklasowej klasyfikacji

In [101]:
# Funkcja do trenowania modelu
def train_modelOczka(model, train_loader, optimizer, criterion, device, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            optimizer.zero_grad()  # Zerowanie gradientów
            outputs = model(inputs)  # Przewidywanie
            loss = criterion(outputs, labels)  # Obliczanie straty

            # Backward pass i optymalizacja
            loss.backward()
            optimizer.step()

            # Statystyki
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {100 * correct / total:.2f}%')

In [102]:
# Funkcja do ewaluacji modelu
def evaluate_model(model, test_loader, device):
    model.eval()  # Tryb ewaluacji
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Accuracy on test data: {100 * correct / total:.2f}%')

In [103]:
# Trenowanie modelu
train_modelOczka(model, train_loader, optimizer, criterion, device, epochs=35) # PRZY 50 MODEL SIĘ PRZEUCZYŁ I ŹLE ODPOWIADAŁ|

Epoch 1/35, Loss: 1.8804, Accuracy: 19.42%
Epoch 2/35, Loss: 1.4768, Accuracy: 38.03%
Epoch 3/35, Loss: 1.1033, Accuracy: 54.92%
Epoch 4/35, Loss: 0.8151, Accuracy: 66.66%
Epoch 5/35, Loss: 0.6633, Accuracy: 72.44%
Epoch 6/35, Loss: 0.5887, Accuracy: 75.66%
Epoch 7/35, Loss: 0.5227, Accuracy: 78.40%
Epoch 8/35, Loss: 0.4769, Accuracy: 80.39%
Epoch 9/35, Loss: 0.4398, Accuracy: 81.92%
Epoch 10/35, Loss: 0.4107, Accuracy: 83.24%
Epoch 11/35, Loss: 0.3826, Accuracy: 84.55%
Epoch 12/35, Loss: 0.3610, Accuracy: 85.51%
Epoch 13/35, Loss: 0.3390, Accuracy: 86.58%
Epoch 14/35, Loss: 0.3243, Accuracy: 87.43%
Epoch 15/35, Loss: 0.3164, Accuracy: 87.50%
Epoch 16/35, Loss: 0.2894, Accuracy: 88.64%
Epoch 17/35, Loss: 0.2787, Accuracy: 89.24%
Epoch 18/35, Loss: 0.2691, Accuracy: 89.63%
Epoch 19/35, Loss: 0.2495, Accuracy: 90.42%
Epoch 20/35, Loss: 0.2487, Accuracy: 90.51%
Epoch 21/35, Loss: 0.2336, Accuracy: 91.07%
Epoch 22/35, Loss: 0.2299, Accuracy: 91.43%
Epoch 23/35, Loss: 0.2115, Accuracy: 92.0

In [104]:
# Ewaluacja modelu
evaluate_model(model, test_loader, device)

Accuracy on test data: 89.94%


In [106]:
X_validity = df.iloc[:, 2:].values.astype('float32')  # piksele
y_validity = df.iloc[:, 0].values.astype('int64')     # czy kostka poprawna (0/1)

scaler_validity = StandardScaler()
X_validity_scaled = scaler_validity.fit_transform(X_validity)

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X_validity_scaled, y_validity, test_size=0.2, random_state=42)

X_train_tensor_v = torch.tensor(X_train_v, dtype=torch.float32)
X_test_tensor_v = torch.tensor(X_test_v, dtype=torch.float32)
y_train_tensor_v = torch.tensor(y_train_v, dtype=torch.long)
y_test_tensor_v = torch.tensor(y_test_v, dtype=torch.long)

train_loader_v = DataLoader(TensorDataset(X_train_tensor_v, y_train_tensor_v), batch_size=32, shuffle=True)
test_loader_v = DataLoader(TensorDataset(X_test_tensor_v, y_test_tensor_v), batch_size=32, shuffle=False)

In [107]:
class ModelPoprawnosci(nn.Module):
    def __init__(self):
        super(ModelPoprawnosci, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 2)  # 2 klasy: poprawna / błędna kostka

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [108]:
validity_model = ModelPoprawnosci()
device = torch.device("cpu")  
validity_model.to(device)

optimizer_v = torch.optim.Adam(validity_model.parameters(), lr=0.001)
criterion_v = nn.CrossEntropyLoss()

# Trening
def train_modelPoprawnosc(model, loader, optimizer, criterion, device, epochs=25): # Epoki zostają, wynik mnie zadowala
    model.train()
    for epoch in range(epochs):
        total_loss, correct, total = 0, 0, 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}, Acc: {correct/total:.4f}")

train_modelPoprawnosc(validity_model, train_loader_v, optimizer_v, criterion_v, device)

Epoch 1, Loss: 0.0607, Acc: 0.9769
Epoch 2, Loss: 0.0205, Acc: 0.9936
Epoch 3, Loss: 0.0160, Acc: 0.9947
Epoch 4, Loss: 0.0171, Acc: 0.9948
Epoch 5, Loss: 0.0106, Acc: 0.9969
Epoch 6, Loss: 0.0132, Acc: 0.9957
Epoch 7, Loss: 0.0105, Acc: 0.9968
Epoch 8, Loss: 0.0079, Acc: 0.9975
Epoch 9, Loss: 0.0095, Acc: 0.9970
Epoch 10, Loss: 0.0081, Acc: 0.9974
Epoch 11, Loss: 0.0080, Acc: 0.9974
Epoch 12, Loss: 0.0081, Acc: 0.9973
Epoch 13, Loss: 0.0066, Acc: 0.9977
Epoch 14, Loss: 0.0076, Acc: 0.9977
Epoch 15, Loss: 0.0065, Acc: 0.9979
Epoch 16, Loss: 0.0076, Acc: 0.9976
Epoch 17, Loss: 0.0065, Acc: 0.9979
Epoch 18, Loss: 0.0054, Acc: 0.9982
Epoch 19, Loss: 0.0058, Acc: 0.9982
Epoch 20, Loss: 0.0064, Acc: 0.9980
Epoch 21, Loss: 0.0053, Acc: 0.9983
Epoch 22, Loss: 0.0052, Acc: 0.9982
Epoch 23, Loss: 0.0060, Acc: 0.9981
Epoch 24, Loss: 0.0043, Acc: 0.9986
Epoch 25, Loss: 0.0054, Acc: 0.9983


In [109]:
def evaluate_validity_model(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    print(f"Accuracy on validity test set: {correct / total:.4f}")

evaluate_validity_model(validity_model, test_loader_v, device)

Accuracy on validity test set: 0.9918


In [110]:
def predict_dice(image_tensor, model, validity_model, device):
    model.eval()
    validity_model.eval()

    with torch.no_grad():
        image_tensor = image_tensor.to(device)

        # Przewidujemy liczbę oczek
        output = model(image_tensor)
        predicted_dots = torch.argmax(output, dim=1).item()

        # Automatyczne odrzucenie dla niepoprawnych oczek
        if predicted_dots in [0, 7, 8, 9]:
            return "Kostka błędna"

        # W przeciwnym razie: sprawdź, czy kostka jest prawidłowa
        validity_output = validity_model(image_tensor)
        validity_pred = torch.argmax(validity_output, dim=1).item()

        if validity_pred == 1:
            return f"Kostka prawidłowa (liczba oczek: {predicted_dots})"
        else:
            return "Kostka błędna"

In [111]:
# Po treningu modelu oczek
torch.save(model.state_dict(), 'simple_nn_model.pth')

# Po treningu modelu poprawności
torch.save(validity_model.state_dict(), 'validity_model.pth')

# Zapis skalera
import joblib
joblib.dump(scaler, 'scaler_dots.pkl')

['scaler_dots.pkl']